In [2]:
!nvidia-smi

Wed Jun 17 13:55:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd "/content/drive/MyDrive/research/external_repos"

/content/drive/MyDrive/research/external_repos


In [5]:
!git clone https://github.com/huggingface/peft

Cloning into 'peft'...
remote: Enumerating objects: 17253, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 17253 (delta 36), reused 16 (delta 16), pack-reused 17181 (from 3)
Receiving objects: 100% (17253/17253), 25.89 MiB | 15.00 MiB/s, done.
Resolving deltas: 100% (12032/12032), done.
Updating files: 100% (863/863), done.


In [6]:
%cd /content/drive/MyDrive/research/external_repos/peft/examples/lora_dreambooth
!ls

/content/drive/MyDrive/research/external_repos/peft/examples/lora_dreambooth
colab_notebook.ipynb		     lora_dreambooth_inference.ipynb
convert_kohya_ss_sd_lora_to_peft.py  requirements.txt
convert_peft_sd_lora_to_kohya_ss.py  train_dreambooth.py


In [8]:
!pip install -r /content/drive/MyDrive/research/external_repos/peft/examples/lora_dreambooth/requirements.txt
!pip install -e /content/drive/MyDrive/research/external_repos/peft

Obtaining file:///content/drive/MyDrive/research/external_repos/peft
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for peft (pyproject.toml) ... done
  Created wheel for peft: filename=peft-0.19.2.dev0-0.editable-py3-none-any.whl size=10853 sha256=39f055f48c3360cbc79d42962d2b328e0ca91d2bbb82006c1dd07f00781567b1
  Stored in directory: /tmp/pip-ephem-wheel-cache-e1s7s1be/wheels/e2/70/7b/e6873b424c6f09652c09a70baece58a7544b78286abf9a84e5
Successfully built peft
  Attempting uninstall: peft
    Found existing installation: peft 0.19.2.dev0
    Uninstalling peft-0.19.2.dev0:
      Successfully uninstalled peft-0.19.2.dev0


In [7]:
!pip install -r requirements.txt
!pip install git+https://github.com/huggingface/peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00
  Cloning https://github.com/huggingface/peft to /tmp/pip-req-build-vkrv1egw
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/peft /tmp/pip-req-build-vkrv1egw
  Resolved https://github.com/huggingface/peft to commit f80d068617012f392580065d6d8119b8518b488d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for peft: filename=peft-0.19.2.dev0-py3-none-any.whl size=722177 sha256=6a0db1d3190898cb53d519556260808b5a76956543a77ee44a23b87460d79fb9
  Stored in directory: /tmp/pip-ephem-wheel-cache-4aaqsfz3/wheels/92/c9/f9/6d1893af725a9b89369aa9416c272f2d512491873dd9db54cd
Successfully built peft
  Attempting uninstall: peft
    Found existing installation: peft 0.19.1
    Uninstalling peft-0.19.1:
      Successfully uninstalled peft-0.19.1


In [14]:
!export MODEL_NAME="CompVis/stable-diffusion-v1-4"
!export PROJECT_DIR="/content/drive/MyDrive/research/lora-dreambooth-project"
!export INSTANCE_DIR="$PROJECT_DIR/datasets/dreambooth/dataset/dog"
!export CLASS_DIR="$PROJECT_DIR/datasets/class_images/dog"
!export OUTPUT_DIR="$PROJECT_DIR/outputs/dog_lora"

In [ ]:
!accelerate launch train_dreambooth.py \
  --pretrained_model_name_or_path=$MODEL_NAME  \
  --instance_data_dir=$INSTANCE_DIR \
  --class_data_dir=$CLASS_DIR \
  --output_dir=$OUTPUT_DIR \
  --train_text_encoder \
  --with_prior_preservation --prior_loss_weight=1.0 \
  --instance_prompt="a photo of sks dog" \
  --class_prompt="a photo of dog" \
  --resolution=512 \
  --train_batch_size=1 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --num_class_images=200 \
  --use_lora \
  --lora_r 16 \
  --lora_alpha 27 \
  --lora_text_encoder_r 16 \
  --lora_text_encoder_alpha 17 \
  --learning_rate=1e-4 \
  --gradient_accumulation_steps=1 \
  --gradient_checkpointing \
  --max_train_steps=800

In [ ]:
import os
import torch

from diffusers import StableDiffusionPipeline
from peft import PeftModel, LoraConfig

MODEL_NAME = "CompVis/stable-diffusion-v1-4"

In [ ]:
def get_lora_sd_pipeline(
    ckpt_dir, base_model_name_or_path=None, dtype=torch.float16, device="cuda", adapter_name="default"
):
    unet_sub_dir = os.path.join(ckpt_dir, "unet")
    text_encoder_sub_dir = os.path.join(ckpt_dir, "text_encoder")
    if os.path.exists(text_encoder_sub_dir) and base_model_name_or_path is None:
        config = LoraConfig.from_pretrained(text_encoder_sub_dir)
        base_model_name_or_path = config.base_model_name_or_path

    if base_model_name_or_path is None:
        raise ValueError("Please specify the base model name or path")

    pipe = StableDiffusionPipeline.from_pretrained(base_model_name_or_path, torch_dtype=dtype).to(device)
    pipe.unet = PeftModel.from_pretrained(pipe.unet, unet_sub_dir, adapter_name=adapter_name)

    if os.path.exists(text_encoder_sub_dir):
        pipe.text_encoder = PeftModel.from_pretrained(
            pipe.text_encoder, text_encoder_sub_dir, adapter_name=adapter_name
        )

    if dtype in (torch.float16, torch.bfloat16):
        pipe.unet.half()
        pipe.text_encoder.half()

    pipe.to(device)
    return pipe

Now you can use the function above to create a Stable Diffusion pipeline using the LoRA weights that you have created during the fine-tuning step.
Note, if you’re running inference on the same machine, the path you specify here will be the same as OUTPUT_DIR.

In [ ]:
pipe = get_lora_sd_pipeline(Path("path-to-saved-model"), adapter_name="dog")

Once you have the pipeline with your fine-tuned model, you can use it to generate images:

In [ ]:
prompt = "sks dog playing fetch in the park"
negative_prompt = "low quality, blurry, unfinished"
image = pipe(prompt, num_inference_steps=50, guidance_scale=7, negative_prompt=negative_prompt).images[0]
image.save("DESTINATION_PATH_FOR_THE_IMAGE")